# Trip Distance Distribution

**Grain:** pickup year/month × pickup borough × distance band. Bands use the source `trip_distance` unit: zero, greater than zero but below 1, 1–<3, 3–<10, and 10 or more. Null and negative distances are excluded because they cannot be assigned a meaningful band. `trip_share_pct` uses all eligible trips in the same borough-month as its denominator, so the band shares sum to approximately 100% (rounding aside). The zero-distance band is retained to make unusual trips visible rather than hiding them among short trips.


In [1]:
from pathlib import Path
import sys

# Find the project even when Jupyter starts in its notebooks directory.
project_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root / "notebooks" / "data_products"))
from product_utils import get_spark, register_integrated, publish_product

spark = get_spark()
register_integrated(spark)


D:\Master\data intensive\data-intensive-computing\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


## Build the product

Run this cell to replace the Delta snapshot. The SQL comments explain the key metric choices.


In [2]:
query = """
WITH classified AS (
    SELECT
        pickup_year,
        pickup_month,
        COALESCE(pickup_borough, 'Unknown') AS pickup_borough,
        trip_id, trip_distance, fare_amount, trip_duration_minutes,
        CASE
            WHEN trip_distance = 0 THEN '00_zero'
            WHEN trip_distance < 1 THEN '01_under_1'
            WHEN trip_distance < 3 THEN '02_1_to_under_3'
            WHEN trip_distance < 10 THEN '03_3_to_under_10'
            ELSE '04_10_or_more'
        END AS distance_band
    FROM integrated_taxi_trips
    WHERE trip_id IS NOT NULL
      AND trip_distance IS NOT NULL
      AND trip_distance >= 0
), band_summary AS (
    SELECT
        pickup_year, pickup_month, pickup_borough, distance_band,
        COUNT(trip_id) AS trip_count,
        ROUND(AVG(trip_distance), 3) AS avg_trip_distance,
        ROUND(AVG(fare_amount), 2) AS avg_base_fare,
        ROUND(AVG(trip_duration_minutes), 2) AS avg_trip_duration_minutes
    FROM classified
    GROUP BY pickup_year, pickup_month, pickup_borough, distance_band
)
SELECT
    pickup_year, pickup_month, pickup_borough, distance_band,
    trip_count,
    ROUND(100.0 * trip_count
          / NULLIF(SUM(trip_count) OVER (
              PARTITION BY pickup_year, pickup_month, pickup_borough
          ), 0), 2) AS trip_share_pct,
    avg_trip_distance, avg_base_fare, avg_trip_duration_minutes
FROM band_summary
"""
result = spark.sql(query)
product = publish_product(spark, result, "trip_distance_distribution")
product.printSchema()
product.show(20, truncate=False)


Saved trip_distance_distribution to D:\Master\data intensive\data-intensive-computing\data\processed\analytical_products\trip_distance_distribution
root
 |-- pickup_year: integer (nullable = true)
 |-- pickup_month: integer (nullable = true)
 |-- pickup_borough: string (nullable = true)
 |-- distance_band: string (nullable = true)
 |-- trip_count: long (nullable = true)
 |-- trip_share_pct: decimal(27,2) (nullable = true)
 |-- avg_trip_distance: double (nullable = true)
 |-- avg_base_fare: double (nullable = true)
 |-- avg_trip_duration_minutes: double (nullable = true)
 |-- source_table: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- refreshed_at: timestamp (nullable = true)
 |-- schema_version: string (nullable = true)

+-----------+------------+--------------+----------------+----------+--------------+-----------------+-------------+-------------------------+---------------------+--------------------------+--------------------------+--------------+
|picku